### data load

In [50]:
import pandas as pd 

df = pd.read_csv(r"..\data\cleaned_data.csv") 

In [51]:
X = df[['channel', 'campaign_type','device', 'user_type', 'region','visited_website','viewed_product','added_to_cart','checkout_started']] 
y = df[['purchase_completed']]

In [52]:
X = pd.get_dummies(X, drop_first=True, dtype=int)
y = pd.get_dummies(y, drop_first=True, dtype=int)

In [53]:
df['added_to_cart'] = df['added_to_cart'].map({'No':0,'Yes':1})
df['checkout_started'] = df['checkout_started'].map({'No':0,'Yes':1})
df['visited_website'] = df['visited_website'].map({'No':0,'Yes':1})
df['viewed_product'] = df['viewed_product'].map({'No':0,'Yes':1}) 
df['purchase_completed'] = df['purchase_completed'].map({'No':0,'Yes':1}) 


bool_cols = df.select_dtypes(include='bool').columns

df[bool_cols] = df[bool_cols].astype(int) 

In [54]:
from sklearn.linear_model import  LogisticRegression
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [55]:
model = LogisticRegression() 

model.fit(X_train,y_train)

y_pred_lr = model.predict(X_test) 

c:\Users\wardo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


### L-Regression

In [56]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,roc_auc_score

print('accuracy score :', accuracy_score(y_test,y_pred_lr))
print('precision score  :', precision_score(y_test,y_pred_lr))
print('recall score :', recall_score(y_test,y_pred_lr))
print('f1 score :', f1_score(y_test,y_pred_lr))
print('ROC AUC score :',roc_auc_score(y_test,y_pred_lr))
print('confusion matrix \n',confusion_matrix(y_test,y_pred_lr))


accuracy score : 0.933625
precision score  : 0.49975161450571287
recall score : 0.6319095477386935
f1 score : 0.5581137309292649
ROC AUC score : 0.7934851201742378
confusion matrix 
 [[21401  1007]
 [  586  1006]]


### Random forest

In [90]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    class_weight='balanced'
)

In [91]:
rf_model.fit(X_train,y_train)

y_prob_rf = rf_model.predict_proba(X_test)[:, 1]
threshold_rf = 0.92
y_pred_rf_threshold = (y_prob_rf >= threshold_rf).astype(int)

c:\Users\wardo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [92]:
print("Accuracy:", accuracy_score(y_test, y_pred_rf_threshold))
print("Precision:", precision_score(y_test, y_pred_rf_threshold))
print("Recall:", recall_score(y_test, y_pred_rf_threshold))
print("F1 Score:", f1_score(y_test, y_pred_rf_threshold))
print('ROC AUC score :',roc_auc_score(y_test,y_prob_rf))
print('confusion matrix \n',confusion_matrix(y_test,y_pred_rf_threshold)) 

Accuracy: 0.9342916666666666
Precision: 0.5028163725122042
Recall: 0.8410804020100503
F1 Score: 0.6293772032902468
ROC AUC score : 0.9640124264664989
confusion matrix 
 [[21084  1324]
 [  253  1339]]


In [98]:
import joblib

joblib.dump(rf_model, "../models/rf_purchase_model.pkl")

['../models/rf_purchase_model.pkl']

### XG-Boost

In [60]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.08,
    random_state=42,
    eval_metric='logloss'
)

In [61]:
xgb_model.fit(X_train,y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

In [62]:
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall:", recall_score(y_test, y_pred_xgb))
print("F1 Score:", f1_score(y_test, y_pred_xgb))
print('ROC AUC score :',roc_auc_score(y_test,y_pred_xgb))
print('confusion matrix \n',confusion_matrix(y_test,y_pred_xgb))

Accuracy: 0.93525
Precision: 0.5089622641509434
Recall: 0.6777638190954773
F1 Score: 0.5813577586206896
ROC AUC score : 0.8156535982303519
confusion matrix 
 [[21367  1041]
 [  513  1079]]


In [63]:
df.columns 

Index(['user_id', 'session_id', 'date', 'month', 'channel', 'campaign_type',
       'device', 'user_type', 'region', 'visited_website', 'viewed_product',
       'added_to_cart', 'checkout_started', 'purchase_completed',
       'discount_applied', 'order_value', 'revenue'],
      dtype='str')

In [64]:
df.head()

,user_id,session_id,date,month,channel,campaign_type,device,user_type,region,visited_website,viewed_product,added_to_cart,checkout_started,purchase_completed,discount_applied,order_value,revenue
0,221958,1,2025-08-16,2025-08,Organic,New Launch,Mobile,New,Metro,1,0,0,0,0,No,499.00,0.000
1,771155,2,2025-12-16,2025-12,Organic,Influencer,Mobile,New,Non-Metro,1,1,1,0,0,No,499.00,0.000
2,231932,3,2025-07-17,2025-07,Organic,Influencer,Mobile,New,Non-Metro,1,1,0,0,0,No,499.00,0.000
3,465838,4,2025-07-04,2025-07,Paid Ads,Discount,Mobile,Returning,Metro,1,1,1,1,1,Yes,2000.95,1800.855
4,359178,5,2025-08-10,2025-08,Paid Ads,Influencer,Mobile,Returning,Non-Metro,1,0,0,0,0,No,499.00,0.000
